In [7]:
#code start here

In [12]:
import json
import urllib.request
from packaging.specifiers import SpecifierSet

# 1. Ensure requirements.txt is written
requirements_content = """django==4.2
numpy==2.3.0
scipy==1.7.0
scikit-learn==1.4.0
fastapi-agent-router==0.3.1
matplotlib==3.99.0
six==1.16.0
requests==2.32.0"""

with open("requirements.txt", "w") as f:
    f.write(requirements_content)

# 2. Check requirements function
def check_requirements(filepath):
    with open(filepath, "r") as f:
        lines = [
            line.strip()
            for line in f
            if line.strip() and not line.strip().startswith("#")
        ]

    checked_count = len(lines)
    not_found_count = 0
    incompatible_count = 0
    highest_req = None
    max_min_version = None

    for line in lines:
        if "==" in line:
            parts = line.split("==")
            name, version = parts[0].strip(), parts[1].strip()
        else:
            continue

        # Check package existence
        url_pkg = f"https://pypi.org/pypi/{name}/json"
        try:
            req_pkg = urllib.request.Request(url_pkg, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req_pkg) as response:
                pkg_data = json.loads(response.read().decode())
        except Exception:
            not_found_count += 1
            continue

        # Check version existence
        url_ver = f"https://pypi.org/pypi/{name}/{version}/json"
        try:
            req_ver = urllib.request.Request(url_ver, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req_ver) as response:
                ver_data = json.loads(response.read().decode())
        except Exception:
            not_found_count += 1
            continue

        # Get requires_python
        info = ver_data.get("info", {})
        requires_python = info.get("requires_python")

        # Compatibility check with Python 3.11
        py_version = "3.11"
        if requires_python:
            spec = SpecifierSet(requires_python)
            if py_version not in spec:
                incompatible_count += 1

            for specifier in spec:
                if specifier.operator in (">=", ">"):
                    try:
                        ver_tuple = tuple(map(int, specifier.version.split(".")))
                        if max_min_version is None or ver_tuple > max_min_version:
                            max_min_version = ver_tuple
                            highest_req = f"{name}=={version}(>={specifier.version})"
                    except ValueError:
                        pass

    print(f"Dependencise checked: {checked_count}")
    print(f"Not found pypi: {not_found_count}")
    print(f"incompatible with python 3.11: {incompatible_count}")
    print(f"Highest requirement: {highest_req}")

if __name__ == "__main__":
    check_requirements("requirements.txt")

Dependencise checked: 8
Not found pypi: 2
incompatible with python 3.11: 1
Highest requirement: numpy==2.3.0(>=3.11)
